In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import burn_scorer
import numpy as np
import pandas as pd

In [ ]:
import burn_scorer
import numpy as np
import pandas as pd

data = []

for _ in range(200):
    temp = np.random.uniform(60, 90)
    humidity = np.random.uniform(10, 70)
    wind = np.random.uniform(0, 25)
    soil = np.random.uniform(5, 40)
    wind_dir = np.random.uniform(0, 360)

    result = burn_scorer.score_burn_window(
        temperature_f=temp,
        humidity_pct=humidity,
        wind_speed_mph=wind,
        soil_moisture_pct=soil,
        wind_direction_deg=wind_dir
    )

    data.append({
        "temperature_f": temp,
        "humidity_pct": humidity,
        "wind_speed_mph": wind,
        "soil_moisture_pct": soil,
        "wind_direction_deg": wind_dir,
        "burn_score": result["burn_score"],
        "recommendation": result["recommendation"]
    })

df = pd.DataFrame(data)
df.head()

In [ ]:
X = df[[
    "temperature_f",
    "humidity_pct",
    "wind_speed_mph",
    "soil_moisture_pct",
    "wind_direction_deg"
]]

y = df["burn_score"]

print(X.head())
print(y.head())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)
print("Model trained.")

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("MAE:", mae)
print("R^2:", r2)

In [ ]:
sample = pd.DataFrame([{
    "temperature_f": 68,
    "humidity_pct": 45,
    "wind_speed_mph": 8,
    "soil_moisture_pct": 32,
    "wind_direction_deg": 180
}])

prediction = model.predict(sample)
print("Predicted burn score:", prediction[0])

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(feature_importance)

plt.figure(figsize=(8,5))
plt.bar(feature_importance["feature"], feature_importance["importance"])
plt.xticks(rotation=30)
plt.ylabel("Importance")
plt.xlabel("Feature")
plt.title("Random Forest Feature Importance")
plt.show()

## ML Baseline: Burn Score Prediction

We trained a Random Forest model on synthetic environmental data generated from our rule-based burn scoring system.

Results:
- Mean Absolute Error (MAE): ~5
- R² Score: ~0.90

This demonstrates that our model can learn meaningful relationships between environmental conditions and burn suitability, providing a foundation for future training on real-world data.